# Amaliyot 14. Klasterlash sifati

14-interaktiv laboratoriyaga («Klasterlash sifatini baholash») hamroh. Kurs mavzusi: siluet, Devis–Bouldin va Kalinski–Xarabas indekslari

Daftar oʻz-oʻzidan yetarli: birinchi katak `common.py` faylini oʻquv korpusi va yordamchi funksiyalar bilan yaratadi. Kataklarni yuqoridan pastga ishga tushiring. Korpus oʻquv (shablon) boʻlib, haqiqiy masalada oʻz CSV faylingizni quyidagi ustunlar bilan qoʻying: `text` va `label`.

In [ ]:
%%writefile common.py
"""Matnlarni tasniflash va klasterlash amaliyotlari uchun umumiy funksiyalar."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "sport": ("oʻyin jamoa gol oʻyinchi murabbiy chempionat turnir gʻalaba magʻlubiyat mavsum stadion muxlis hakam hisob "
              "himoyachi hujumchi darvozabon kubok liga final transfer klub zarba penalti musobaqa sportchi "
              "rekord medal olimpiada mashq durang").split(),
    "siyosat": ("qonun deputat saylov partiya ovoz prezident vazir hukumat parlament islohot byudjet "
                "sanksiya nomzod koalitsiya vazirlik farmon tuzatish loyiha muxolifat referendum diplomatiya "
                "hokim konstitutsiya majlis komissiya muzokara sammit saylovchi mandat vakolat").split(),
    "oshxona": ("shoʻrva retsept tuz goʻsht pishirmoq duxovka sabzavot tova qozon xamir shakar ziravor qaynatmoq qovurmoq "
                "dimlamoq sous bulyon salat oshpaz taom masalliq yogʻ un tuxum sut sarimsoq piyoz garmdori "
                "kartoshka shirinlik pishiriq marinad").split(),
    "texnologiya": ("protsessor dastur ilova smartfon neyrotarmoq algoritm maʼlumot server yangilanish dasturchi "
                    "kompyuter xotira ekran kamera brauzer tarmoq internet bulut kod platforma foydalanuvchi qurilma "
                    "chip batareya noutbuk dasturiy videokarta tizim interfeys shifrlash").split(),
}
NEUTRAL = ("bugun juda yangi aytishadi keyin ham mumkin edi boʻldi maʼlum qilindi kecha yaqinda hali hammasi "
           "bir nechta muhim birinchi").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """4 mavzudagi qisqa «hujjatlar»dan iborat oʻquv korpusi.
    noise: oʻz mavzusidan boʻlmagan soʻzlar ulushi (neytral yoki begona). noise qancha katta boʻlsa, masala shuncha qiyin."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """TF-IDF siyrak matritsasini tekislikka proyeksiyalash (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Har bir sinf uchun eng katta vaznli soʻzlarni chiqaradi (chiziqli modellar)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))


In [ ]:
from common import *

# Oʻquv korpusi: 4 ta mavzu. noise — "begona" soʻzlar ulushi (qancha koʻp boʻlsa, shuncha qiyin).
df = make_corpus(n_per_class=150, noise=0.45, seed=0)
# Oʻquv korpusi oʻrniga oʻzingiznikini qoʻying: df = pd.read_csv("texts.csv")  # ustunlar: text, label
print(df.label.value_counts().to_dict())
df.head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score

X = TfidfVectorizer().fit_transform(df.text); y = df.label.values
Xr = make_pipeline(TruncatedSVD(30, random_state=0), Normalizer(copy=False)).fit_transform(X)

## 1. k = 2 dan 10 gacha uchta metrika

In [ ]:
ks = range(2, 11); rows = []
for k in ks:
    lab = KMeans(k, n_init=10, random_state=0).fit_predict(Xr)
    rows.append((k, silhouette_score(Xr, lab), davies_bouldin_score(Xr, lab), calinski_harabasz_score(Xr, lab), adjusted_rand_score(y, lab)))
res = pd.DataFrame(rows, columns=["k", "silhouette", "davies_bouldin", "calinski_harabasz", "ARI"]).set_index("k")
fig, ax = plt.subplots(1, 4, figsize=(16, 3.5))
for a, (c, t) in zip(ax, [("silhouette", "Siluet (kattaroq yaxshiroq)"), ("davies_bouldin", "Devis-Bouldin (kichikroq yaxshiroq)"), ("calinski_harabasz", "Kalinski-Xarabas (kattaroq yaxshiroq)"), ("ARI", "Haqiqiy mavzular bilan ARI")]):
    res[c].plot(marker="o", ax=a, title=t)
plt.tight_layout(); plt.show()
res.round(3)

## 2. Tanlangan k uchun siluet grafigi

In [ ]:
k = 4
lab = KMeans(k, n_init=10, random_state=0).fit_predict(Xr)
s = silhouette_samples(Xr, lab)
y0 = 0; plt.figure(figsize=(6, 5))
for c in range(k):
    v = np.sort(s[lab == c]); plt.barh(range(y0, y0 + len(v)), v, height=1.0, label=f"klaster {c}"); y0 += len(v) + 10
plt.axvline(s.mean(), color="red", ls="--"); plt.xlabel("siluet"); plt.title(f"k={k}, oʻrtacha siluet {s.mean():.3f}"); plt.legend(); plt.show()

## Topshiriqlar

1. Qiymatini oshiring: `noise` — 0.7 gacha. Metrikalar qanday oʻzgaradi va ular oʻzaro mos keladimi?
2. DBSCAN yoki aglomerativ klasterlash uchun metrikalarni hisoblang. Qaysi metrika ularga «mos kelmaydi» va nima uchun?
3. Ichki metrikalar va ARI boʻyicha k tanlovini solishtiring. ARI qachon mavjud emas?